### Imports


In [1]:
from pprint import pprint
from tqdm.auto import tqdm
from haystack.nodes import QuestionGenerator, BM25Retriever, FARMReader
from haystack.document_stores import ElasticsearchDocumentStore
from haystack.pipelines import (
    QuestionGenerationPipeline,
    RetrieverQuestionGenerationPipeline,
    QuestionAnswerGenerationPipeline,
)
from haystack.utils import launch_es, print_questions, add_example_data
from haystack import Pipeline
from haystack.document_stores import InMemoryDocumentStore

2023-10-14 22:03:52.345690: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Logging configuration


In [2]:
import logging

logging.basicConfig(
    format="%(levelname)s - %(name)s -  %(message)s", level=logging.WARNING
)
logging.getLogger("haystack").setLevel(logging.INFO)

### Question Generator


In [ ]:
document_store = InMemoryDocumentStore()
text1 = "Stories are short descriptions of a small piece of desired functionality written from the user’s perspective. Agile Teams implement stories as small, vertical slices of system functionality that can be completed in a few days or less. Stories are the primary artifact used to define system behavior in Agile. They are short, simple descriptions of functionality told from the user’s perspective and written in their language. Each implements a small, vertical slice of system behavior. Stories provide just enough information for business and technical people to understand the intent. Details are deferred until the story is ready to be implemented. Through acceptance criteria and acceptance tests, stories get more specific, helping to ensure system quality. User stories deliver functionality directly to the end user. Enabler stories bring visibility to the work items needed to support exploration, architecture, infrastructure, and compliance."
text2 = "Iteration planning is a SAFe Scrum event where all team members determine how much of the Team Backlog they can commit to delivering during an upcoming Iteration. The team summarizes this work as a set of committed iteration goals. Iteration planning is the first event of the Iteration. During planning, the team defines, organizes, and commits to the work for the next iteration. The iteration planning meeting is timeboxed to approximately 90 minutes for a two-week iteration. The team’s backlog has been partially identified and planned during PI Planning. In addition, the teams have feedback—not only from their prior iterations but also from the System Demo, stakeholders, and others. All this context feeds into the iteration planning event to inform the plan for the upcoming iteration."
text3 = "An Epic is a significant solution development initiative. Due to their considerable scope and impact, epics require the definition of a Minimum Viable Product (MVP) and approval by Lean Portfolio Management (LPM). Portfolio epics are typically cross-cutting, typically spanning multiple Value Streams and PIs. To accelerate learning and development and reduce risk, SAFe recommends applying the Lean Startup build-measure-learn cycle for these epics. There are two types of epics, each of which may occur at different levels of the Framework. Business epics directly deliver business value, while enabler epics advance the Architectural Runway to support upcoming business or technical needs."

docs = [{"content": text1}, {"content": text2}, {"content": text3}]
document_store.write_documents(docs)

In [ ]:
question_generator = QuestionGenerator()
question_generation_pipeline = QuestionGenerationPipeline(question_generator)
for idx, document in enumerate(document_store):
    print(
        f"\n * Generating questions for document {idx}: {document.content[:100]}...\n"
    )
    result = question_generation_pipeline.run(documents=[document])
    print_questions(result)

### Question and Answer Generator


In [17]:
document_store = InMemoryDocumentStore()
add_example_data(document_store, "data/")
pprint(document_store.get_document_count())

INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
INFO - haystack.utils.getting_started -  Adding 6 number of files from local disk at data/.
INFO - haystack.utils.preprocessing -  Converting data/Epic.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_4.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_3.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_2.txt
INFO - haystack.utils.preprocessing -  Converting data/Story_1.txt
INFO - haystack.utils.preprocessing -  Converting data/Iteration_Planning.txt
Preprocessing: 100%|██████████| 6/6 [00:00<00:00, 1500.38docs/s]

6


In [18]:
question_generator = QuestionGenerator()
reader = FARMReader("deepset/roberta-base-squad2")
question_answer_generation_pipeline = QuestionAnswerGenerationPipeline(
    question_generator, reader
)
for idx, document in enumerate(tqdm(document_store)):
    print(
        f"\n * Generating questions and answers for document {idx}: {document.content[:100]}...\n"
    )
    result = question_answer_generation_pipeline.run(documents=[document])
    print_questions(result)

INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
Using sep_token, but it is not set yet.
INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0
INFO - haystack.modeling.model.language_model -   * LOADING MODEL: 'deepset/roberta-base-squad2' (Roberta)
INFO - haystack.modeling.model.language_model -  Auto-detected model language: english
INFO - haystack.modeling.model.language_model -  Loaded 'deepset/roberta-base-squad2' (Roberta model) from model hub.
INFO - haystack.modeling.utils -  Using devices: CPU - Number of GPUs: 0


0it [00:00, ?it/s]


 * Generating questions and answers for document 0: An Epic is a significant solution development initiative. Due to their considerable scope and impact...



Inferencing Samples: 100%|██████████| 1/1 [00:01<00:00,  1.51s/ Batches]



Generated pairs:
 - Q: What is a significant solution development initiative?
      A: An Epic
 - Q: Epics require the definition of what?
      A: Minimum Viable Product
 - Q: What are portfolio epics typically cross-cutting?
      A: multiple Value Streams and PIs
 - Q: How many types of epics are there?
      A: two
 - Q: What cycle does SAFe recommend for epics?
      A: Lean Startup build-measure-learn cycle
 - Q: Business epics directly deliver what kind of value?
      A: business
 - Q: Business epics directly deliver what?
      A: business value
 - Q: Enabler epics advance what to support upcoming business or technical needs?
      A: Architectural Runway

 * Generating questions and answers for document 1: While anyone can write stories, approving them into the team backlog and accepting them into the sys...



...


Generated pairs:
 - Q: What are the four tiers of artifacts that outline functional system behavior called?
      A: Epic, Capability, Feature, and Story
 - Q: What are Epic, Capability, Feature, and Story used to describe collectively?
      A: the solution’s intended behavior
 - Q: The detailed implementation work is expressed through stories, which comprise what?
      A: Team Backlog
 - Q: What are some stories that emerge from business and enabler features in the ART Backlog?
      A: stories emerge from business and enabler features in the ART Backlog, while others come from the team’s local context

 * Generating questions and answers for document 4: Stories are short descriptions of a small piece of desired functionality written from the user’s per...



Inferencing Samples: 100%|██████████| 1/1 [00:01<00:00,  1.85s/ Batches]



Generated pairs:
 - Q: What are short descriptions of a small piece of desired functionality written from the user's perspective?
      A: Stories
 - Q: Agile Teams implement stories as small, vertical slices of what?
      A: system functionality
 - Q: Stories are the primary artifact used to define what in Agile?
      A: system behavior
 - Q: What is used to define system behavior in Agile?
      A: Stories
 - Q: What are artifacts short, simple descriptions of?
      A: functionality
 - Q: How are details deferred until the story is ready to be implemented?
      A: Stories provide just enough information for business and technical people to understand the intent. Details are deferred until the story is ready to be implemented. Through acceptance criteria and acceptance tests
 - Q: Through acceptance criteria and acceptance tests, stories get more specific, helping to ensure what?
      A: system quality
 - Q: User stories deliver functionality directly to the end user.
      A: S

Inferencing Samples: 100%|██████████| 1/1 [00:01<00:00,  1.56s/ Batches]


Generated pairs:
 - Q: What is a SAFe Scrum event where all team members determine how much of the Team Backlog they can commit to deliver during an upcoming iteration?
      A: Iteration planning
 - Q: What does the team summarize this work as?
      A: a set of committed iteration goals
 - Q: Iteration planning is the first event of what?
      A: Iteration
 - Q: What is the first event of an Iteration?
      A: Iteration planning
 - Q: What does the team define, organize, and commit to during planning?
      A: the work for the next iteration
 - Q: How long is the iteration planning meeting?
      A: 90 minutes
 - Q: During PI Planning, what has been partially identified and planned?
      A: The team’s backlog
 - Q: What does all this context feed into the iteration planning event?
      A: to inform the plan for the upcoming iteration
